# 实验六补充 2：小语言模型微调实验

这份 notebook 面向“完全没有接触过微调”的同学。我们会从最朴素的问题开始：模型已经会续写文本了，为什么还要微调？微调到底是在改模型的知识，还是在改模型的行为？训练时 loss 是什么？为什么小数据会过拟合？

默认模型是 `distilbert/distilgpt2`。它是 GPT-2 的蒸馏版本，Hugging Face 模型卡标注 Apache-2.0，约 88M 参数。它不是聊天模型，英语能力也有限，但正因为它小，才适合在课堂上把微调过程完整跑一遍。

本实验采用 soft prompt tuning：冻结原模型，只训练一小段可学习的“虚拟 token embedding”。这不是工业界唯一的微调方式，但很适合教学，因为它让你清楚看到：微调可以只改很少的参数，也能明显改变模型输出风格。

教师版已经填入答案。发布学生版时，`START CODE HERE` 与 `END CODE HERE` 之间的代码会被挖空。本 notebook 只有 3 处需要学生补代码，其余代码以演示和观察为主。


## 0. 从零理解：什么是微调

大语言模型训练通常分成几个阶段。不同课程和论文叫法略有差异，但核心逻辑相同：

1. **预训练**：模型读大量普通文本，学习“给定前文预测下一个 token”。这一步让模型学到语言模式、常识片段、代码格式等。
2. **监督微调**：把数据整理成任务格式，例如“问题 -> 答案”“指令 -> 回复”，让模型模仿我们想要的回答方式。
3. **偏好对齐**：用人类偏好或自动偏好信号进一步调整，让模型更符合“有帮助、诚实、安全”等目标。

本实验只做第二步的一种迷你版本。请注意两个边界：

- 微调不是魔法。小模型、小数据、短训练无法得到真正可靠的问答助手。
- 微调很容易改变模型行为。哪怕只训练很少参数，模型也可能更偏向某种输出模板、语气或答案长度。

你可以把微调想象成：预训练模型已经学会了很多“语言反射”，微调数据会告诉它“在这个场景下，请优先使用这一类反射”。


## 1. 环境与设备

微调需要反向传播，比单纯推理更吃内存。本实验冻结原模型，只训练 soft prompt，因此大部分电脑都能跑。CPU 可以运行；Apple Silicon 会尝试 MPS；NVIDIA GPU 会使用 CUDA。

如果缺少依赖，先取消下一格的 `%pip install` 注释并运行。安装后重启 kernel。


In [ ]:
# 如缺少依赖，取消下一行注释并运行一次；安装后重启 kernel。
# %pip install -U torch transformers accelerate safetensors matplotlib

import importlib.util
missing = [pkg for pkg in ["torch", "transformers", "matplotlib"] if importlib.util.find_spec(pkg) is None]
if missing:
    print("缺少依赖：", missing)
    print("请先运行：%pip install -U torch transformers accelerate safetensors matplotlib")
else:
    print("依赖检查通过。")


**输出说明**

如果看到“依赖检查通过”，说明当前 `pt` kernel 中已经有本实验需要的主要库。如果这里提示缺少依赖，请先安装并重启 kernel；否则后面模型加载或训练会失败。


In [ ]:
import math
import random
from copy import deepcopy

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "distilbert/distilgpt2"

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print("device:", device)


**输出说明**

这里显示的是本机实际使用的计算设备。`cuda` 通常最快，`mps` 是 Apple Silicon 的 GPU 后端，`cpu` 最通用但训练会慢一些。本实验训练参数很少，所以即使用 CPU 也能作为教学演示运行。


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
base_model.eval()

# GPT-2 系列通常没有 pad token。为了 batch padding，课堂实验中直接复用 eos token。
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded {MODEL_ID}")
print(f"parameters: {n_params / 1e6:.1f}M")
print("vocab size:", len(tokenizer))


**输出说明**

这里会打印模型参数量和词表大小。参数量表示模型内部可学习权重的规模；词表大小表示 tokenizer 能输出多少种 token id。后面 soft prompt tuning 只会训练几千个参数，对比 8000 多万参数的原模型非常少。


## 2. 微调前：先看看原模型会怎么回答

在微调前，`distilgpt2` 只是一个普通续写模型。它不一定理解 `Question:` / `Answer:` 是问答格式，也不一定会给出课程风格的解释。先观察原模型的输出，可以帮助我们理解微调前后的区别。

注意：这里不是在评价模型“聪明不聪明”，而是在观察它的默认行为分布。微调的目标就是把这个分布往我们的小数据集格式上推一点。


In [ ]:
@torch.no_grad()
def generate_with_base(prompt, max_new_tokens=35, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = base_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-6),
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

for prompt in [
    "Question: What is overfitting?\nAnswer:",
    "Question: What is a token?\nAnswer:",
]:
    print("\nPROMPT:", prompt)
    print(generate_with_base(prompt, max_new_tokens=35, temperature=0.7))


**输出说明**

你通常会看到：原模型能续写英语，但回答未必准确、简洁或符合课程定义。这说明“会续写”和“会按任务格式回答”是两件不同的事。微调不是从零教模型语言，而是用少量样本告诉它：在这种模板下，请更像课程助教那样回答。


## 3. 微调数据：模型到底在学什么

自回归语言模型的训练目标仍然是 next-token prediction。所谓“监督微调”，通常只是把训练文本组织成我们希望模型模仿的格式。例如：

```text
Question: What is overfitting?
Answer: Overfitting means the model fits training data too closely and generalizes poorly.<eos>
```

模型不会天然知道哪一段是问题、哪一段是答案。我们通过文本模板告诉它：看见 `Question:` 后面是用户输入，看见 `Answer:` 后面应该续写回答。

一个重要细节是 **label masking**。我们希望模型学习“答案怎么写”，而不是浪费损失去背问题本身。因此会把 prompt 部分的 label 设成 `-100`。在 Hugging Face 的 causal language model 中，`-100` 表示这个位置不计入 loss。

### 练习 1：把问答样本变成训练样本

你只需要补 4 行：构造 prompt、构造完整文本、tokenize 完整文本、计算 prompt 的 token 长度。后面的 label masking 已经写好。


In [ ]:
train_examples = [
    {"question": "What is overfitting?", "answer": "Overfitting means a model fits training data too closely and performs poorly on new data."},
    {"question": "What is a validation set?", "answer": "A validation set is held-out data used to tune choices without touching the test set."},
    {"question": "What is gradient descent?", "answer": "Gradient descent updates parameters in the direction that reduces the loss."},
    {"question": "What is attention?", "answer": "Attention lets each token mix information from relevant tokens in the context."},
    {"question": "Why use regularization?", "answer": "Regularization discourages overly complex solutions and can improve generalization."},
    {"question": "What is fine-tuning?", "answer": "Fine-tuning adapts a pretrained model to a smaller task-specific dataset."},
    {"question": "What is a learning rate?", "answer": "A learning rate controls how large each parameter update is during optimization."},
    {"question": "What is a token?", "answer": "A token is a text piece represented by an integer id before it enters the model."},
]

MAX_LEN = 96

def encode_example(example):
    ### START CODE HERE ###
    # TODO 1：写出模型看到的问题模板。
    # TODO 2：把答案和 eos token 接到 prompt 后面。
    # TODO 3：tokenize 完整文本。
    # TODO 4：单独 tokenize prompt，用长度来屏蔽 prompt loss。
    ### END CODE HERE ###

    labels = encoded["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len
    encoded["labels"] = labels
    return encoded

encoded_train = [encode_example(ex) for ex in train_examples]
print("num examples:", len(encoded_train))
print("first input length:", len(encoded_train[0]["input_ids"]))
print("ignored label positions:", sum(x == -100 for x in encoded_train[0]["labels"]))


**输出说明**

`num examples` 是训练样本数；`first input length` 是第一个样本 token 化后的长度；`ignored label positions` 是被 mask 掉的 prompt token 数量。这些位置仍然作为上下文输入模型，但不计算训练损失。也就是说，模型看见问题，但只因为答案部分受到惩罚或奖励。


In [ ]:
def show_label_mask(encoded, max_tokens=40):
    ids = encoded["input_ids"][:max_tokens]
    labels = encoded["labels"][:max_tokens]
    for pos, (token_id, label) in enumerate(zip(ids, labels)):
        token_text = tokenizer.decode([token_id]).replace("\n", "\\n")
        target = "ignored" if label == -100 else "learn"
        print(f"{pos:02d} | {target:7s} | {token_text!r}")

show_label_mask(encoded_train[0])


**输出说明**

这张小表把每个 token 标成 `ignored` 或 `learn`。`ignored` 的 prompt token 只提供条件；`learn` 的答案 token 才参与 loss。这个设计很常见：如果不 mask prompt，模型会花一部分能力去预测固定模板和问题，而不是专注学习答案格式。


In [ ]:
def collate_batch(features):
    max_len = max(len(f["input_ids"]) for f in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for f in features:
        pad = max_len - len(f["input_ids"])
        batch["input_ids"].append(f["input_ids"] + [tokenizer.pad_token_id] * pad)
        batch["attention_mask"].append([1] * len(f["input_ids"]) + [0] * pad)
        batch["labels"].append(f["labels"] + [-100] * pad)
    return {k: torch.tensor(v, dtype=torch.long, device=device) for k, v in batch.items()}

batch = collate_batch(encoded_train[:3])
print({k: tuple(v.shape) for k, v in batch.items()})
assert batch["input_ids"].shape == batch["labels"].shape
assert (batch["labels"] == -100).sum() > 0
print("数据编码检查通过。")


**输出说明**

这里的形状通常是 `[batch_size, sequence_length]`。同一个 batch 内样本长度不同，所以要 padding 到同一长度。`attention_mask` 告诉模型哪些位置是真 token，哪些位置只是 padding；`labels` 中 padding 位置也设为 `-100`，避免 padding 参与 loss。


## 4. 微调方式：全量微调与参数高效微调

全量微调会更新模型所有参数。优点是表达能力强；缺点是显存、时间、数据量要求都高，而且容易过拟合。在真实大模型场景中，常见做法是参数高效微调（PEFT），例如 LoRA、adapter、prefix tuning、prompt tuning。

本实验使用 soft prompt tuning。它的想法是：不改原模型，而是在输入 token embedding 前面拼接几个可学习向量。它们不是人类可读 token，但会通过反向传播学到“如何引导模型回答”。

数学上，如果原始输入 embedding 是：

$$
E = [e_1, e_2, \ldots, e_n]
$$

soft prompt tuning 会变成：

$$
E' = [p_1, p_2, \ldots, p_m, e_1, e_2, \ldots, e_n]
$$

其中 $p_1, \ldots, p_m$ 是可训练参数，原模型参数被冻结。这样训练量很小，适合课堂演示。

### 练习 2：实现 soft prompt 前向传播

你只需要补 5 行：取 token embedding、扩展 soft prompt、拼接 embedding、拼接 attention mask、拼接 labels。注意 labels 前面要补 `-100`，因为 soft prompt 没有真实目标 token。


In [ ]:
class SoftPromptTuningLM(nn.Module):
    def __init__(self, base_model, soft_prompt_tokens=12):
        super().__init__()
        self.base = base_model
        for p in self.base.parameters():
            p.requires_grad_(False)
        hidden_size = self.base.config.n_embd
        self.soft_prompt = nn.Parameter(torch.randn(soft_prompt_tokens, hidden_size) * 0.02)

    def forward(self, input_ids, attention_mask=None, labels=None):
        batch_size = input_ids.shape[0]
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        ### START CODE HERE ###
        # TODO 1：把 token id 变成原模型的 token embedding。
        # TODO 2：把 soft prompt 扩展到 batch 维度。
        # TODO 3：在真实 token embedding 前面拼接 soft prompt。
        # TODO 4：attention_mask 前面补 1，表示 soft prompt 可见。
        # TODO 5：labels 前面补 -100，表示 soft prompt 不参与 loss。
        ### END CODE HERE ###

        return self.base(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)

SOFT_PROMPT_TOKENS = 12
prompt_model = SoftPromptTuningLM(base_model, soft_prompt_tokens=SOFT_PROMPT_TOKENS).to(device)
initial_soft_prompt = prompt_model.soft_prompt.detach().clone()
trainable = sum(p.numel() for p in prompt_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in prompt_model.parameters())
print(f"soft prompt shape: {tuple(prompt_model.soft_prompt.shape)}")
print(f"trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")


**输出说明**

`soft prompt shape` 的第二维是模型隐藏维度，第一维是虚拟 token 个数。可训练参数比例通常非常小，这就是参数高效微调的核心：训练成本低、保存也方便。但它的表达能力也有限，不能指望它像全量微调一样大幅改变模型能力。


In [ ]:
@torch.no_grad()
def average_loss(model, encoded_examples):
    losses = []
    for ex in encoded_examples:
        b = collate_batch([ex])
        losses.append(float(model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], labels=b["labels"]).loss.item()))
    return sum(losses) / len(losses)

initial_loss = average_loss(prompt_model, encoded_train)
print(f"average training loss before tuning: {initial_loss:.4f}")
print(f"approx perplexity before tuning: {math.exp(min(initial_loss, 20)):.1f}")


**输出说明**

这是训练前的平均 loss 和粗略困惑度。困惑度可以理解为“模型平均每一步有多少个差不多的选择”。数值越低，说明模型越容易预测训练答案。微调后我们会再看 loss 是否下降。


## 5. 训练循环：loss 下降意味着什么

训练循环的核心步骤非常固定：前向传播得到 loss，反向传播计算梯度，优化器更新参数，清空梯度。这里我们只训练 `soft_prompt`，所以优化器里只有很少的参数。

loss 下降说明模型在训练样本上更容易预测答案 token。但这并不等于它真正理解了概念。数据很小时，loss 可以很快下降，同时模型也可能只是记住了格式或训练答案。因此微调实验一定要同时观察：

- 训练 loss 是否下降。
- 未见过问题的生成是否更接近目标格式。
- 改变学习率、步数、soft prompt 长度时，结果是否稳定。

### 练习 3：补全一次训练 step

你只需要补 5 行：前向传播、取 loss、反向传播、更新参数、清空梯度。


In [ ]:
LR = 5e-2
NUM_STEPS = 25
BATCH_SIZE = 4
optimizer = torch.optim.AdamW([prompt_model.soft_prompt], lr=LR)
loss_history = []

prompt_model.train()
for step in range(NUM_STEPS):
    batch_examples = random.sample(encoded_train, k=BATCH_SIZE)
    batch = collate_batch(batch_examples)

    ### START CODE HERE ###
    # TODO 1：前向传播，传入 input_ids、attention_mask、labels。
    # TODO 2：读取 loss。
    # TODO 3：反向传播。
    # TODO 4：参数更新。
    # TODO 5：清空梯度。
    ### END CODE HERE ###

    loss_history.append(float(loss.item()))
    if step % 5 == 0 or step == NUM_STEPS - 1:
        print(f"step {step:02d} | loss {loss.item():.4f}")

prompt_model.eval()
final_loss = average_loss(prompt_model, encoded_train)
print("first sampled-batch loss:", round(loss_history[0], 4))
print("last sampled-batch loss:", round(loss_history[-1], 4))
print("average training loss after tuning:", round(final_loss, 4))
assert math.isfinite(loss_history[-1])


**输出说明**

如果训练有效，loss 往往会整体下降，但不一定每一步都下降，因为每次随机抽到的 batch 不同。`average training loss after tuning` 更稳定一些。请记住：训练集 loss 下降只说明模型更适应这些训练样本，不自动代表泛化能力变强。


In [ ]:
delta = (prompt_model.soft_prompt.detach() - initial_soft_prompt).norm().item()
print(f"soft prompt parameter movement L2 norm: {delta:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(loss_history, marker="o")
plt.xlabel("training step")
plt.ylabel("batch loss")
plt.title("Soft prompt tuning loss")
plt.grid(True, alpha=0.3)
plt.show()


**输出说明**

L2 norm 表示 soft prompt 参数相对初始化移动了多少。loss 曲线展示训练过程是否稳定。如果曲线大幅震荡，可能是学习率过大；如果几乎不动，可能是学习率太小、训练步数太少，或者 soft prompt 表达能力不足。


## 6. 微调后：输出真的变了吗

接下来比较同一个 prompt 在原模型和 soft prompt 微调模型下的输出。你不应该期待小模型给出完美答案；更重要的是观察格式、用词、答案倾向是否发生变化。


In [ ]:
@torch.no_grad()
def generate_with_soft_prompt(prompt_model, prompt, max_new_tokens=40, temperature=0.7):
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    for _ in range(max_new_tokens):
        logits = prompt_model(input_ids=input_ids).logits[0, -1]
        if temperature == 0:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token.reshape(1, 1)], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

test_prompt = "Question: What is fine-tuning?\nAnswer:"
print("--- before / base model ---")
print(generate_with_base(test_prompt, max_new_tokens=35, temperature=0.7))
print("\n--- after / soft prompt tuned ---")
print(generate_with_soft_prompt(prompt_model, test_prompt, max_new_tokens=35, temperature=0.7))


**输出说明**

这里最值得观察的是“风格迁移”：微调后模型是否更倾向于用定义句回答？是否更像训练集里的答案？如果答案仍然重复或跑题，这不是实验失败，而是小模型、小数据和短训练的真实局限。


In [ ]:
for question in [
    "What is a token?",
    "What is gradient descent?",
    "Why do models overfit?",
]:
    prompt = f"Question: {question}\nAnswer:"
    print("\nPROMPT:", prompt)
    print(generate_with_soft_prompt(prompt_model, prompt, max_new_tokens=35, temperature=0.3))


**输出说明**

把 temperature 降低后，输出会更保守、更容易重复训练集中常见的表达。你可以看到微调带来的“有趣之处”：模型不只是背某一句话，而是整个回答模式都被推向了训练数据的风格。但也能看到风险：数据太少时，模型可能学会空泛模板，而不是稳健知识。


## 7. 可选小实验：自己改参数观察

你可以在上面重新运行时改这些变量：

- `SOFT_PROMPT_TOKENS`：虚拟 token 越多，可训练参数越多；太少可能表达不够，太多可能更容易过拟合。
- `LR`：学习率太小，loss 降得慢；学习率太大，loss 可能震荡甚至变坏。
- `NUM_STEPS`：步数越多，训练集 loss 往往越低，但不代表泛化更好。
- `temperature`：生成时的随机性。微调后的模型如果仍然胡说，可以先把 temperature 降低。

建议实验报告记录两组设置：一组 loss 降得比较稳，一组故意把学习率调大或步数调少。比较它们的 loss 曲线和生成结果。


## 8. 总结问题

1. 微调数据中的文本模板为什么重要？如果把 `Question:` 和 `Answer:` 去掉，会发生什么？
2. 为什么要把 prompt 部分 label 设成 `-100`？
3. soft prompt tuning 和全量微调相比，牺牲了什么，换来了什么？
4. loss 下降一定代表模型更有用吗？请结合你看到的生成结果回答。
5. 如果要把这个实验扩展成真正可用的课程问答助手，还需要哪些数据、评估和安全检查？
